# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bivo2004/my-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis & Time Window

* **Unit of Analysis:** One row equals one uniquely published web page (content piece) belonging to a specific client.
* **Time Window:** The performance metrics (like impressions and CTR) represent an aggregated snapshot over a **90-day window**. The data does not provide a daily time-series; it is a point-in-time aggregate of the last 3 months.

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Setup Colab Environment
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    if not os.getcwd().endswith("flyrank-ml-internship-starter"):
        os.chdir("flyrank-ml-internship-starter")

# 2. Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 3. Verify grain
total_rows = len(df)
unique_pages = df["content_id"].nunique()

print(f"Total Rows: {total_rows}")
print(f"Unique Content IDs: {unique_pages}")
print(f"Is the grain strictly 1 row = 1 page? {total_rows == unique_pages}")

Total Rows: 30000
Unique Content IDs: 30000
Is the grain strictly 1 row = 1 page? True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Sorting

* **Context:** `content_id`, `client_id` (Used to track items and split train/test sets properly so one client's data doesn't leak into the test set. Not used for training).
* **Label:** `trend_direction` (Observed outcome to map to `is_declining`).
* **Features:** `days_since_last_update`, `impressions_90d`, `search_volume`, `avg_position`, `ctr`, `word_count`.
* **Excluded:**
  * `trend_pct` (Massive Leakage: This is the exact mathematical percentage used to create our label. Including it would let the model cheat).
  * `competition_level` (Redundant: We already capture this information in the numerical `competition` column).

In [2]:
# Define our contract lists
context = ["content_id", "client_id"]
label = ["trend_direction"]
features = ["days_since_last_update", "impressions_90d", "search_volume", "avg_position", "ctr", "word_count"]
excluded = ["trend_pct", "competition_level"]

# Verify all claimed columns actually exist in the dataframe
all_claimed = context + label + features + excluded
missing_cols = [col for col in all_claimed if col not in df.columns]

print(f"Are all contract columns present in the dataset? {len(missing_cols) == 0}")
if missing_cols:
    print(f"Missing: {missing_cols}")

Are all contract columns present in the dataset? True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Queries

We must verify three things:
1. **Grain Integrity:** No duplicate `content_id` rows.
2. **Missing Values:** Ensure our core features are populated enough to train a model.
3. **Label Health:** Ensure the target label has a balanced enough distribution to actually learn from.

In [3]:
# 1. Grain Integrity Check
is_grain_valid = df["content_id"].duplicated().sum() == 0
print(f"1. Zero duplicate content IDs: {is_grain_valid}")

# 2. Missing Value Check (Null counts for our features)
print("\n2. Missing values in selected features:")
print(df[features].isna().sum())

# 3. Label Health Check
print("\n3. Target Label Distribution:")
print(df["trend_direction"].value_counts(dropna=False))

1. Zero duplicate content IDs: True

2. Missing values in selected features:
days_since_last_update       0
impressions_90d              0
search_volume             2468
avg_position                 0
ctr                          0
word_count                7699
dtype: int64

3. Target Label Distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations & Boundaries

1. **Snapshot, not Time-Series:** We only see 90-day aggregates (e.g., `impressions_90d`). We cannot pinpoint *which exact week* a page started decaying, only that it is down over a 90-day window.
2. **No Causality:** We observe that a page is declining, but the data cannot tell us *why*. It could be a Google algorithm update, a competitor publishing better content, or seasonal search drops.
3. **Traffic Bias:** Impressions and CTR are likely constrained to search engine data (like Google Search Console) and do not account for direct or social media traffic health.

In [4]:
# Prove the time-series limitation by checking available time columns
time_cols = [c for c in df.columns if "date" in c.lower() or "day" in c.lower()]

print("Available time-related columns:")
for col in time_cols:
    print(f"- {col}")

print("\nConclusion: The dataset only contains static snapshots (like 'days_since_last_update' or 'impressions_90d').")
print("There are no daily timestamps to build a time-series model, verifying our limitation claim.")

Available time-related columns:
- days_with_impressions
- days_with_sessions
- content_age_days
- days_since_last_update

Conclusion: The dataset only contains static snapshots (like 'days_since_last_update' or 'impressions_90d').
There are no daily timestamps to build a time-series model, verifying our limitation claim.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.